In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
# sample documents
documents = [
    "This is a list which contains sample documents",
    "Keywords are important for keyword-based search",
    "Document analysis involves extracting keywords",
    "Keyword-based search relies on sparse embeddings",
]

In [3]:
# keyword-based search
keyword = "keyword-based search"

In [4]:
import re


def preprocess_text(text):
    """converts text to lowercase & removes punctuations"""
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text


print(preprocess_text(documents[3]))

keywordbased search relies on sparse embeddings


In [5]:
preprocessed_documents = [preprocess_text(doc) for doc in documents]
preprocessed_documents

['this is a list which contains sample documents',
 'keywords are important for keywordbased search',
 'document analysis involves extracting keywords',
 'keywordbased search relies on sparse embeddings']

In [6]:
# vectoirize all our documents
vectorizer = TfidfVectorizer()
docx = vectorizer.fit_transform(preprocessed_documents)
docx[0].toarray()

array([[0.        , 0.        , 0.37796447, 0.        , 0.37796447,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.37796447, 0.        , 0.        , 0.37796447, 0.        ,
        0.        , 0.37796447, 0.        , 0.        , 0.37796447,
        0.37796447]])

In [7]:
# use the same vectorizer to vectorize our keyword too
queryx = vectorizer.transform([preprocess_text(keyword)])
queryx.toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.70710678, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.70710678, 0.        , 0.        ,
        0.        ]])

In [8]:
similarities = cosine_similarity(docx, queryx)
similarities
# from the result below, you can see that docx[1] has the most similarity with the query

array([[0.        ],
       [0.50551777],
       [0.        ],
       [0.48693426]])

In [9]:
# now let's get the document
best_match_doc = documents[np.argmax(similarities)]
best_match_doc

'Keywords are important for keyword-based search'

In [10]:
# get all documents from best matching to worst
[documents[i] for i in np.argsort(similarities, axis=0).flatten()[::-1]]

['Keywords are important for keyword-based search',
 'Keyword-based search relies on sparse embeddings',
 'Document analysis involves extracting keywords',
 'This is a list which contains sample documents']

In [11]:
# now let's use an embedding from Hugging Face's sentence transformer library
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(documents)
print(embeddings)


[[ 0.0224453  -0.04584402 -0.00341779 ...  0.00063181 -0.05187821
   0.0092258 ]
 [ 0.05577597 -0.02369128 -0.03720743 ...  0.02876143 -0.06962651
  -0.00014712]
 [ 0.05780773 -0.01864415 -0.02694583 ... -0.01305465 -0.08049827
   0.01200785]
 [ 0.04056801  0.00330962  0.01014766 ...  0.01633128 -0.13285087
  -0.00915645]]


### Now let's use LangChain framework for hybrid search

In [12]:
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [13]:
pdf_path = Path('papers/Retrieval-Augmented-Generation-for-NLP.pdf')
loader = PyPDFLoader(str(pdf_path))
pdf_docs = loader.load()
print(f"Loaded {len(pdf_docs)} documents")

Loaded 19 documents


In [14]:
# display a random document from loaded documents
index = np.random.randint(0, len(pdf_docs))
print(f"doc[{index}] = {pdf_docs[index]}")

doc[11] = page_content='The sentiment analysis task is crucial for understanding consumer
feedback, monitoring brand reputation, and gaining insights into
public opinion on various issues.
RAG techniques can significantly enhance sentiment analysis
with different external knowledge fusion strategies. Li et al. [98]
concatenate the retrieved options and corresponding prompt-based
labels with input options. Other works [ 16, 52] concatenate the
retrieval embeddings with input embeddings before feeding them
into the decoder. Some works fuse the retrieval features into the
hidden states of generators via cross-attention [17, 163] or ranking-
based addition [169]. Besides, other works focus on fusing the logits
of retrievals with the output logit using ensemble techniques [179,
185]. Except for knowledge fusions, Min et al. [118] enable locating
knowledge in phrases more accurately via two queries.
8.7 Dialogue Systems
Dialogue systems, also known as conversational agents or chatbots,
are d

In [15]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
splits = splitter.split_documents(pdf_docs)
print(f"Got {len(splits)} chunks")

Got 810 chunks


In [16]:
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain_community.vectorstores import FAISS
import os

In [17]:
embeddings = HuggingFaceInferenceAPIEmbeddings(
    api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    model_name="BAAI/bge-base-en-v1.5"
)
faiss_store = Path(os.getcwd()) / "faiss_index_rag4nlp"

vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
retriever = vector_store.as_retriever()
vector_store.save_local(str(faiss_store))

C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_48644\2493111206.py:1: LangChainDeprecationWarning: The class `HuggingFaceInferenceAPIEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpointEmbeddings``.
  embeddings = HuggingFaceInferenceAPIEmbeddings(


KeyError: 0